In [0]:
from pyspark.sql.functions import col, monotonically_increasing_id, round, to_date

In [0]:
pay_df = spark.table("01_global_tech_bronze.raw_tables.payroll_raw").drop("gross_salary", "net_salary")
emp_df = spark.table("01_global_tech_bronze.raw_tables.employee_raw").select("employee_id", "base_salary")


In [0]:
transformed_pay_df = (
    pay_df
    .withColumn("bonus", col("bonus").cast("double"))
    .withColumn("overtime_pay", col("overtime_pay").cast("double"))
    .withColumn("commission", col("commission").cast("double"))
    .withColumn("allowances", col("allowances").cast("double"))
    .withColumn("pay_date", to_date(col("pay_date"), "dd-MM-yyyy HH:mm"))
    .withColumn("pay_period_start", to_date(col("pay_date"), "dd-MM-yyyy HH:mm"))
    .withColumn("pay_period_end", to_date(col("pay_date"), "dd-MM-yyyy HH:mm"))
    .withColumn("total_compensation", round((col("bonus") + col("overtime_pay") + col("commission") + col("allowances")), 2))
    .withColumn("total_deduction", round((col("tax_deduction") + col("social_security") + col("health_insurance") + col("retirement_contribution") +
                                          col("other_deductions")), 2))
    .withColumnRenamed("status", "payroll_status")
)

transformed_pay_df = (
    transformed_pay_df.join(emp_df, on="employee_id", how="left")
    .withColumn("net_salary", round(
        col("base_salary") + col("total_compensation") - col("total_deduction"), 2))
    .drop("base_salary").withColumn("payroll_sk", monotonically_increasing_id())
)

transformed_pay_df.write.format("delta").mode("overwrite").saveAsTable("02_global_tech_silver.transformed_tables.transformed_payroll")
# display(transformed_pay_df)

In [0]:
# display(transformed_pay_df)